# 04 - Feature Engineering

**Project:** Customer Financial Behavior Analysis for Banking Business Intelligence

Pada tahap ini dibuat fitur-fitur baru agar karakteristik pelanggan dapat direpresentasikan dengan lebih baik sebelum dilakukan preprocessing dan clustering.


In [ ]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv("bank_transactions_clean.csv")

# Konversi tanggal
df["TransactionDate"] = pd.to_datetime(df["TransactionDate"])
df["PreviousTransactionDate"] = pd.to_datetime(df["PreviousTransactionDate"])

df.head()


,TransactionID,AccountID,TransactionAmount,PreviousTransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,TransactionDate
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70.0,Doctor,81.0,1.0,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68.0,Doctor,141.0,1.0,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19.0,Student,56.0,1.0,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26.0,Student,25.0,1.0,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,45.0,Student,198.0,1.0,7429.40,2024-11-04 08:06:39


In [ ]:
# Transaction Hour
df["TransactionHour"] = df["TransactionDate"].dt.hour

# Day of Week
df["DayOfWeek"] = df["TransactionDate"].dt.day_name()

# Days Since Previous Transaction
df["DaysSincePreviousTransaction"] = (
    df["TransactionDate"] - df["PreviousTransactionDate"]
).dt.days

df[["TransactionHour","DayOfWeek","DaysSincePreviousTransaction"]].head()


,TransactionHour,DayOfWeek,DaysSincePreviousTransaction
0,8,Monday,572
1,8,Monday,495
2,8,Monday,482
3,8,Monday,548
4,8,Monday,384


In [ ]:
# Total Transaction per Account
total_transaction = (
    df.groupby("AccountID")["TransactionID"]
      .count()
      .rename("TotalTransactionPerAccount")
)

# Average Transaction
average_transaction = (
    df.groupby("AccountID")["TransactionAmount"]
      .mean()
      .rename("AverageTransaction")
)

# Average Balance
average_balance = (
    df.groupby("AccountID")["AccountBalance"]
      .mean()
      .rename("AverageBalance")
)


In [ ]:
# Debit Count
debit_count = (
    df[df["TransactionType"]=="Debit"]
    .groupby("AccountID")
    .size()
    .rename("DebitCount")
)

# Credit Count
credit_count = (
    df[df["TransactionType"]=="Credit"]
    .groupby("AccountID")
    .size()
    .rename("CreditCount")
)


In [ ]:
# Favorite Channel
favorite_channel = (
    df.groupby("AccountID")["Channel"]
      .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
      .rename("FavoriteChannel")
)

# Favorite Merchant
favorite_merchant = (
    df.groupby("AccountID")["MerchantID"]
      .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
      .rename("FavoriteMerchant")
)


In [ ]:
# Customer-level Dataset
customer_df = (
    df.groupby("AccountID")
      .agg({
          "CustomerAge":"first",
          "CustomerOccupation":"first",
          "TransactionAmount":"mean",
          "AccountBalance":"mean",
          "LoginAttempts":"mean",
          "TransactionDuration":"mean",
          "TransactionHour":"mean",
          "DaysSincePreviousTransaction":"mean"
      })
)

customer_df = customer_df.join(total_transaction)
customer_df = customer_df.join(average_transaction)
customer_df = customer_df.join(average_balance)
customer_df = customer_df.join(debit_count)
customer_df = customer_df.join(credit_count)
customer_df = customer_df.join(favorite_channel)
customer_df = customer_df.join(favorite_merchant)

customer_df = customer_df.fillna(0)
customer_df.head()


,CustomerAge,CustomerOccupation,TransactionAmount,AccountBalance,LoginAttempts,TransactionDuration,TransactionHour,DaysSincePreviousTransaction,TotalTransactionPerAccount,AverageTransaction,AverageBalance,DebitCount,CreditCount,FavoriteChannel,FavoriteMerchant
AccountID,,,,,,,,,,,,,,,
AC00001,25.0,Student,130.380000,2915.160000,1.000000,107.500000,8.0,385.000000,2,130.380000,2915.160000,2.0,0.0,Branch,M003
AC00002,71.0,Retired,293.744286,4480.421429,1.142857,109.857143,8.0,509.571429,7,293.744286,4480.421429,6.0,1.0,Branch,M040
AC00003,69.0,Doctor,253.268000,7329.568000,1.000000,49.200000,8.0,575.800000,5,253.268000,7329.568000,4.0,1.0,ATM,M026
AC00004,64.0,Retired,264.807500,2712.905000,1.000000,115.000000,8.0,427.875000,8,264.807500,2712.905000,7.0,1.0,Branch,M011
AC00005,52.0,Engineer,347.974444,4457.837778,1.000000,145.111111,8.0,488.777778,9,347.974444,4457.837778,8.0,1.0,Branch,M002


In [ ]:
# Debit & Credit Ratio
customer_df["DebitRatio"] = (
    customer_df["DebitCount"] /
    (customer_df["DebitCount"] + customer_df["CreditCount"] + 1e-6)
)

customer_df["CreditRatio"] = (
    customer_df["CreditCount"] /
    (customer_df["DebitCount"] + customer_df["CreditCount"] + 1e-6)
)

customer_df[["DebitRatio","CreditRatio"]].head()


,DebitRatio,CreditRatio
AccountID,,
AC00001,1.000000,0.000000
AC00002,0.857143,0.142857
AC00003,0.800000,0.200000
AC00004,0.875000,0.125000
AC00005,0.888889,0.111111


In [ ]:
# Save Feature Engineered Dataset
customer_df.to_csv("customer_features.csv")
print("customer_features.csv berhasil dibuat.")
customer_df.head()


customer_features.csv berhasil dibuat.


,CustomerAge,CustomerOccupation,TransactionAmount,AccountBalance,LoginAttempts,TransactionDuration,TransactionHour,DaysSincePreviousTransaction,TotalTransactionPerAccount,AverageTransaction,AverageBalance,DebitCount,CreditCount,FavoriteChannel,FavoriteMerchant,DebitRatio,CreditRatio
AccountID,,,,,,,,,,,,,,,,,
AC00001,25.0,Student,130.380000,2915.160000,1.000000,107.500000,8.0,385.000000,2,130.380000,2915.160000,2.0,0.0,Branch,M003,1.000000,0.000000
AC00002,71.0,Retired,293.744286,4480.421429,1.142857,109.857143,8.0,509.571429,7,293.744286,4480.421429,6.0,1.0,Branch,M040,0.857143,0.142857
AC00003,69.0,Doctor,253.268000,7329.568000,1.000000,49.200000,8.0,575.800000,5,253.268000,7329.568000,4.0,1.0,ATM,M026,0.800000,0.200000
AC00004,64.0,Retired,264.807500,2712.905000,1.000000,115.000000,8.0,427.875000,8,264.807500,2712.905000,7.0,1.0,Branch,M011,0.875000,0.125000
AC00005,52.0,Engineer,347.974444,4457.837778,1.000000,145.111111,8.0,488.777778,9,347.974444,4457.837778,8.0,1.0,Branch,M002,0.888889,0.111111


## Output

Dataset **customer_features.csv** akan digunakan pada tahap berikutnya yaitu **06_Data_Preprocessing.ipynb** untuk proses encoding, scaling, feature selection, dan persiapan clustering.
